# Marketing Mix Modeling: Channel Contributions and Budget Implications

## Scenario
Following the MMM baseline, extract channel-level contribution signals from the model coefficients and translate them into a directional budget recommendation for the meal kit growth team.

## Your task
1. A ranked channel coefficient table sorted by value
2. A horizontal bar chart of channel coefficients with a reference line at zero
3. A written directional budget recommendation

## Data: `mealkit_marketing_weekly.csv`

**Required sentence (verbatim) in your recommendation:**
> *"MMM results are correlational (directional), not causal."*

**Note:** Extract only the adstock column coefficients — exclude t, sin52, cos52.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path('Exercise_Data')
if not DATA_DIR.exists():
    DATA_DIR = Path('../Exercise_Data')

CHOICE_PATH = DATA_DIR / 'mealkit_choice_tasks.csv'
PLANS_PATH  = DATA_DIR / 'mealkit_plans.csv'
ADOPT_PATH  = DATA_DIR / 'mealkit_trial_adoption.csv'
SURVEY_PATH = DATA_DIR / 'mealkit_survey.csv'
MMM_PATH    = DATA_DIR / 'mealkit_marketing_weekly.csv'
from sklearn.linear_model import Ridge

In [ ]:
# Refit MMM — same setup as the baseline exercise
# (This cell is pre-filled so you can proceed directly to coefficient extraction)
def apply_adstock(series, alpha=0.3):
    result = np.zeros(len(series))
    result[0] = series.iloc[0]
    for i in range(1, len(series)):
        result[i] = series.iloc[i] + alpha * result[i-1]
    return pd.Series(result, index=series.index)

df = pd.read_csv(MMM_PATH)
df['date'] = pd.to_datetime(df['date'])
spend_cols   = ['spend_search','spend_social','spend_influencer','spend_email','spend_affiliate']
adstock_cols = [c.replace('spend_','adstock_') for c in spend_cols]
for sc, ac in zip(spend_cols, adstock_cols):
    df[ac] = apply_adstock(df[sc], alpha=0.3)
df['t']     = range(1, len(df) + 1)
df['sin52'] = np.sin(2 * np.pi * df['t'] / 52)
df['cos52'] = np.cos(2 * np.pi * df['t'] / 52)
feat_cols = adstock_cols + ['t', 'sin52', 'cos52']
split_idx = int(len(df) * 0.80)
train = df.iloc[:split_idx]
model = Ridge(alpha=1.0)
model.fit(train[feat_cols].astype(float), train['weekly_revenue'].astype(float))
print("Adstock columns created:", adstock_cols)
print("Time controls added. Total features:", len(feat_cols))
print("Model refitted.")

In [ ]:
# ── Step 1: Extract and rank channel coefficients ────────────────────────
# TODO: Slice model.coef_[:len(adstock_cols)] to get the 5 channel coefficients
#       (The remaining entries in model.coef_ belong to the time controls — exclude them)
# TODO: Build a pd.DataFrame with columns ['channel','coefficient']
# TODO: Sort by coefficient value, descending
# TODO: Print the table

In [ ]:
# ── Step 2: Channel coefficient bar chart ────────────────────────────────
# TODO: Create a horizontal bar chart (ax.barh) of channel coefficients
#       Blue for positive coefficients, red for negative
# TODO: Add a vertical reference line at x=0
# TODO: Label x-axis: 'Coefficient (revenue per adstock unit)'
# TODO: Add title: 'MMM Channel Coefficients'
# TODO: plt.tight_layout(); plt.show()

## Budget Recommendation

**Channels to prioritize (positive coefficients):**
[YOUR ANSWER — name the top 1–2 channels and reference their coefficient values]

**Channels to flag for investigation (negative coefficients):**
[YOUR ANSWER — name them; explain why a negative coefficient does not necessarily mean the channel is harmful — reference counter-cyclical deployment as one possible explanation]

*"MMM results are correlational (directional), not causal."*

**Recommended validation step before making large budget reallocations:**
[YOUR ANSWER — describe a specific holdout test or geo-split experiment]
